# An-Ra Sovereign AI System — Master Control Notebook

```text
Drive <-> Colab <-> GitHub <-> FastAPI <-> User
  |         |         |          |
checkpoints train   sync      /chat,/generate
sessions    test    pull      ngrok tunnel
```


In [ ]:
# Cell 2 — SETUP
from pathlib import Path
import json, os, platform, shutil, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
cfg_path = Path('/content/drive/MyDrive/AnRa/.secrets.json')
cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
if 'REPO_URL' not in cfg:
    cfg['REPO_URL'] = input('Enter REPO_URL: ').strip()
if 'NGROK_TOKEN' not in cfg:
    cfg['NGROK_TOKEN'] = input('Enter NGROK_TOKEN: ').strip()
cfg_path.parent.mkdir(parents=True, exist_ok=True)
cfg_path.write_text(json.dumps(cfg, indent=2))
REPO_URL = cfg['REPO_URL']
NGROK_TOKEN = cfg['NGROK_TOKEN']
REPO_DIR = Path('/content/repo')
if (REPO_DIR / '.git').exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull'])
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'fastapi', 'uvicorn', 'pyngrok', 'httpx'])
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available():
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print('RAM (GB):', round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 2))
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)


In [ ]:
# Cell 3 — SESSION RESTORE
from pathlib import Path
import shutil
DRIVE_ROOT = Path('/content/drive/MyDrive/AnRa')
REPO_DIR = Path('/content/repo')
SESSION_RESTORED = False
CHECKPOINT_SOURCE = None
summary = {}
for ckpt in ['anra_brain_identity.pt', 'anra_brain.pt']:
    src = DRIVE_ROOT / ckpt
    if src.exists():
        shutil.copy2(src, REPO_DIR / ckpt)
        CHECKPOINT_SOURCE = ckpt
        SESSION_RESTORED = True
        summary[ckpt] = 'restored'
        if ckpt == 'anra_brain_identity.pt':
            break
    else:
        summary[ckpt] = 'missing'
tok = DRIVE_ROOT / 'tokenizer.pkl'
if tok.exists():
    shutil.copy2(tok, REPO_DIR / 'tokenizer.pkl')
    summary['tokenizer.pkl'] = 'restored'
else:
    summary['tokenizer.pkl'] = 'missing'
sess_src = DRIVE_ROOT / 'sessions'
if sess_src.exists():
    sess_dst = REPO_DIR / 'sessions'
    sess_dst.mkdir(parents=True, exist_ok=True)
    for p in sess_src.glob('*.json'):
        shutil.copy2(p, sess_dst / p.name)
    summary['sessions'] = 'restored'
else:
    summary['sessions'] = 'missing'
print('SESSION_RESTORED =', SESSION_RESTORED)
print('CHECKPOINT_SOURCE =', CHECKPOINT_SOURCE)
print('Restore summary:', summary)


In [ ]:
# Cell 4 — BASE TRAINING (conditional)
import subprocess, sys, shutil
from pathlib import Path
REPO_DIR = Path('/content/repo')
DRIVE_ROOT = Path('/content/drive/MyDrive/AnRa')
if SESSION_RESTORED:
    print('Checkpoint found — skipping base training')
else:
    subprocess.check_call([sys.executable, 'build_anra_brain.py', '--data_path', 'combined_identity_data.txt', '--checkpoint_path', 'anra_brain.pt'], cwd=REPO_DIR)
    shutil.copy2(REPO_DIR / 'anra_brain.pt', DRIVE_ROOT / 'anra_brain.pt')
    print('Base checkpoint saved to Drive')


In [ ]:
# Cell 5 — IDENTITY FINE-TUNING (conditional)
import subprocess, sys, shutil
from pathlib import Path
REPO_DIR = Path('/content/repo')
DRIVE_ROOT = Path('/content/drive/MyDrive/AnRa')
assert (REPO_DIR / 'data/combined_identity_data.txt').exists() or (REPO_DIR / 'combined_identity_data.txt').exists(), 'data missing'
existing = DRIVE_ROOT / 'anra_brain_identity.pt'
if existing.exists():
    choice = input('Identity checkpoint exists. Type retrain to retrain; anything else skips: ').strip().lower()
    if choice == 'retrain':
        subprocess.check_call([sys.executable, 'finetune_anra.py'], cwd=REPO_DIR)
    else:
        print('Skipping fine-tune')
else:
    subprocess.check_call([sys.executable, 'finetune_anra.py'], cwd=REPO_DIR)
if (REPO_DIR / 'anra_brain_identity.pt').exists():
    shutil.copy2(REPO_DIR / 'anra_brain_identity.pt', DRIVE_ROOT / 'anra_brain_identity.pt')
    print('Identity checkpoint copied to Drive')


In [ ]:
# Cell 6 — LAUNCH API (health-poll strategy)
import os, subprocess, time, requests
from pyngrok import ngrok
os.environ['PYTHONPATH'] = '/content/repo'
proc = subprocess.Popen(['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'], cwd='/content/repo', stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
ready = False
for _ in range(20):
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if r.status_code == 200:
            ready = True
            print('Health:', r.json())
            break
    except Exception:
        pass
    time.sleep(2)
if not ready:
    err = proc.stderr.read()
    raise RuntimeError(f'Server failed health check. stderr:
{err}')
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000).public_url
print('PUBLIC URL:', public_url)


In [ ]:
# Cell 7 — LIVE CHAT (inline)
import requests
api = 'http://127.0.0.1:8000'
session_id = 'notebook_session'
strategy = 'nucleus'
while True:
    msg = input('You: ').strip()
    if msg.lower() == 'quit':
        break
    if msg.lower() == 'reset':
        requests.post(f'{api}/reset', json={'session_id': session_id}, timeout=10)
        print('Session reset')
        continue
    if msg.lower().startswith('switch '):
        strategy = msg.split(' ', 1)[1].strip()
        print('Strategy:', strategy)
        continue
    try:
        r = requests.post(f'{api}/chat', json={'session_id': session_id, 'message': msg, 'params': {'strategy': strategy, 'max_new_tokens': 120}}, timeout=60)
        d = r.json()
        print('An-Ra:', d['reply'])
        print('Turn:', d['turn'])
    except Exception as e:
        print('Connection error:', e)


In [ ]:
# Cell 8 — MODEL COMPARISON
from pathlib import Path
import pickle, torch, torch.nn.functional as F
from anra_brain import CausalTransformer
repo = Path('/content/repo')
tok = pickle.load(open(repo / 'tokenizer.pkl', 'rb'))
def load_model(ckpt):
    m = CausalTransformer(tok.vocab_size, 256, 4, 4, 128)
    s = torch.load(ckpt, map_location='cpu')
    if isinstance(s, dict) and 'model_state_dict' in s:
        s = s['model_state_dict']
    m.load_state_dict(s, strict=False)
    m.eval()
    return m
base = repo / 'anra_brain.pt'
ident = repo / 'anra_brain_identity.pt'
prompts = ['I am', 'My purpose is', 'An-Ra is', 'Who created you', 'What should I do?']
if base.exists() and ident.exists():
    from generate import generate
    print('PROMPT | BASE | IDENTITY')
    print('-'*120)
    for p in prompts:
        b = generate(f'H: {p}\nANRA:', strategy='nucleus', max_new_tokens=80)
        i = generate(f'H: {p}\nANRA:', strategy='nucleus', max_new_tokens=80)
        print(p, '|', b[:40], '|', i[:40])
    bm, im = load_model(base), load_model(ident)
    text = (repo / 'combined_identity_data.txt').read_text(encoding='utf-8', errors='ignore')[:4000]
    ids = torch.tensor(tok.encode(text[:1000]), dtype=torch.long).unsqueeze(0)
    def ppl(m):
        with torch.no_grad():
            logits,_ = m(ids[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), ids[:, 1:].reshape(-1))
        return float(torch.exp(loss))
    base_ppl, ident_ppl = ppl(bm), ppl(im)
    print('Perplexity delta (base-ident):', base_ppl - ident_ppl)
else:
    print('Need both anra_brain.pt and anra_brain_identity.pt for comparison')


In [ ]:
# Cell 9 — SAVE & SHUTDOWN
from pathlib import Path
import shutil, datetime
repo = Path('/content/repo')
drive = Path('/content/drive/MyDrive/AnRa')
drive.mkdir(parents=True, exist_ok=True)
for f in ['anra_brain.pt', 'anra_brain_identity.pt', 'tokenizer.pkl']:
    src = repo / f
    if src.exists():
        shutil.copy2(src, drive / f)
        print(f, 'saved', src.stat().st_size, 'bytes')
sess = repo / 'sessions'
if sess.exists():
    dst = drive / 'sessions'
    dst.mkdir(parents=True, exist_ok=True)
    for p in sess.glob('*.json'):
        shutil.copy2(p, dst / p.name)
print('Saved at', datetime.datetime.utcnow().isoformat() + 'Z')


## Cell 10 — Integration tests


In [ ]:
!python /content/repo/test_suite.py


## Cell 12 — API quick check


In [ ]:
import requests
print(requests.get('http://127.0.0.1:8000/health', timeout=5).json())


## Cell 14 — Docs


In [ ]:
print('Open /docs on the local or ngrok URL.')
